# Classification Challenge — KNN / GaussianNB / LDA / AdaBoost

**Dataset:** Gas Sensor Array Drift (UCI ML Repository id=270) — 13,910 instancias × 128 sensores, 6 clases de gases.
Vergara et al. 2012. Licencia CC-BY-4.0, uso solo investigación.
Tarea aplicada: clasificación de riesgo químico por sensores (encaja en "chemical risk + sensor data" de la rúbrica).

**Modelos (1 por familia mínimo):** Familia 1 → KNN · Familia 2 → GaussianNB + LDA (se repite F2: con 4 modelos y 3 familias una familia debe repetirse) · Familia 3 → AdaBoost.

Clases: 1 Ethanol (2565), 2 Ethylene (2926), 3 Ammonia (1641), 4 Acetaldehyde (1936), 5 Acetone (3009), 6 Toluene (1833).

## Phase 1 — Matriz teórica (principio, outliers, coste, supuestos)

| Criterio | KNN (F1: geométrica/distancias) | GaussianNB (F2: probabilística/generativa) | LDA (F2: discriminante) | AdaBoost (F3: ensemble/estocástica) |
|---|---|---|---|---|
| Principio matemático central | Voto mayoritario (o ponderado por distancia) entre los k vecinos más cercanos según métrica d(x,x'). Sin modelo explícito: frontera de Voronoi local. | Teorema de Bayes + verosimilitud gaussiana por feature con **supuesto de independencia condicional**: P(x\|y)=∏ᵢ N(xᵢ; μᵢ,σᵢ²). | Proyección lineal que maximiza separación entre medias respecto a covarianza común: supone cada clase ~ N(μₖ, Σ) con **Σ compartida** → frontera lineal. Equivale a Bayes gaussiano con covarianzas iguales. | Suma ponderada de clasificadores débiles (stumps/profundidad 1-2): F(x)=Σₜ αₜhₜ(x). Cada ronda repondera muestras mal clasificadas (énfasis adaptativo en errores). |
| Sensibilidad a outliers | **Alta.** Un vecino atípico vota directamente; k pequeño (k=3 ganó) es más sensible. Mitiga con k mayor, weights=distance, métrica manhattan. | **Media.** Media/varianza por feature se sesgan con atípicos; una σ subestimada desploma la likelihood. Mitiga con var_smoothing. | **Alta.** Medias μₖ y covarianza Σ se estiman por MLE y son sensibles a atípicos; un punto extremo rota la frontera lineal. Mitiga con shrinkage. | **Muy alta.** AdaBoost insiste en los puntos difíciles/ruido (les sube el peso cada ronda) → sobreajusta a ruido y drift si T grande. Mitiga con lr<1, T moderado, profundidad baja. |
| Coste computacional | Train O(1) (solo memoriza). Predict O(n·d) por query (13k×128 aquí → predict test 1.8 s, el más lento en inferencia). Memoria O(n·d). | Train O(n·d), predict O(c·d) — el más barato (fit GridSearch 0.3 s). | Train O(n·d² + d³) aprox. por covarianza/inversión (128-D manejable: 0.4 s). Predict O(c·d). | Train O(T·n·d·coste_débil) — el más caro (125 s con T=200, d=2). Predict O(T·profundidad). |
| Suposiciones sobre datos | **Ninguna distribucional** (no paramétrico). Asume que cercanía en el espacio escalado ≈ misma clase; sufre en alta-D dispersa y exige escalado. | Features **gaussianas e independientes** dada la clase. Falla si hay correlación fuerte (aquí 128 sensores correlacionados → F1 0.58). Requiere escalado moderado (NB es invariante a escala en teoría, pero el pipeline unificado lo incluye). | Cada clase **gaussiana con igual covarianza** → frontera lineal óptima. Robusto si se cumple aprox.; falla con covarianzas muy distintas (ahí QDA sería mejor) o multimodalidad. Exige escalado para estabilidad numérica (solver lsqr). | **Ninguna distribucional**, solo que el débil supere al azar. Asume señal aprendible por combinación aditiva; sensible a ruido/etiquetas malas y a desbalance (requiere stratify). |

**Por qué se repite F2:** la rúbrica pide 4 modelos con ≥1 por familia (3 familias) → una familia debe repetirse. Se repite F2 (NB vs LDA) porque contrasta dos usos opuestos del mismo supuesto gaussiano: ingenuo-independiente (NB) vs. discriminante con covarianza compartida (LDA). El experimento confirma la lección: LDA 0.948 vs NB 0.579.

## Phase 2 — Pipeline unificado + Grids + CV

1. **Mismo CSV** `data/gas_drift.csv` (parseado de `data/uci270/batch*.dat` con `src/download_data.py` + renombrado de features con `src/clean_data.py`), mismo `train_test_split(80/20, stratify, random_state=42)` → test n=2782.
2. **Mismo `Pipeline([StandardScaler, clf])`** — scaler ajustado solo en train dentro de cada fold (sin leakage). Sin OneHot (todo numérico).
3. **Mismo `GridSearchCV(cv=Stratified 5-Fold, scoring=f1_macro)`** + `cross_validate` externo 5-Fold (ver `src/experiment.py`).

| Modelo | Grid buscado | Mejor encontrado |
|---|---|---|
| KNN | k[3,5,11,21] × weights[uniform,distance] × metric[euclidean,manhattan] (16 combos) | manhattan, k=3, distance |
| GaussianNB | var_smoothing[1e-9,1e-8,1e-7] | 1e-9 |
| LDA | solver[lsqr,eigen] × shrinkage[None,auto] | lsqr, sin shrinkage |
| AdaBoost (DecisionTree) | n_estimators[50,100,200] × lr[0.5,1.0] × depth[1,2] (12 combos) | 200 árboles d=2, lr=1.0 |

**Re-ejecutar todo (local):** `python src/download_data.py` → `python src/clean_data.py` → `python src/experiment.py` (genera `results/*.csv` + `figures/*.png`).
En Streamlit Cloud **no** se reentrena: la app solo lee resultados precalculados.

### Limpieza de datos: renombrado de features

El dataset crudo trae columnas anónimas `f1..f128` (mediciones de 16 sensores químicos, sin encabezados oficiales). Para que el ojo humano entienda el dataset, se renombran con la regla:

`f(i)` → `Sensor{(i-1)//8+1:02d}_R{(i-1)%8+1}` — es decir, **16 sensores × 8 lecturas**.

| Original | Nuevo | Descripción |
|---|---|---|
| f1..f8 | Sensor01_R1..Sensor01_R8 | 8 lecturas del sensor 1 |
| f9..f16 | Sensor02_R1..Sensor02_R8 | 8 lecturas del sensor 2 |
| ... | ... | ... |
| f121..f128 | Sensor16_R1..Sensor16_R8 | 8 lecturas del sensor 16 |
| class | class | Clase de gas objetivo (1..6) |

Generado con `python src/clean_data.py` (solo estándar-library). El mapeo completo queda en `data/feature_dictionary.csv`.

In [ ]:
# Limpieza ya aplicada: verificar cabecera legible (16 sensores x 8 lecturas)
import pandas as pd
df = pd.read_csv('../data/gas_drift.csv')
print('Cabecera (primeras 10 cols):')
print(df.columns[:10].tolist())
print('\nUltimas cols:')
print(df.columns[-3:].tolist())
print('\nDiccionario (primeras 5):')
print(pd.read_csv('../data/feature_dictionary.csv').head(5).to_string(index=False))

In [ ]:
import pandas as pd
df = pd.read_csv('../data/gas_drift.csv')
print('Shape:', df.shape, '(exigido >= 2500 instancias)')
print(df['class'].value_counts().sort_index().to_dict())

In [ ]:
# Resultados del experimento unificado (test 20% n=2782 + CV externo 5-Fold)
import pandas as pd
s = pd.read_csv('../results/metrics_summary.csv')
print(s[['model','cv_f1_macro_mean','test_acc','test_f1_macro','fit_time_s']].to_string(index=False))
print()
print(pd.read_csv('../results/cv_folds.csv').pivot(index='fold', columns='model', values='f1_macro').round(4).to_string())

In [ ]:
from IPython.display import Image
Image(filename='../figures/f1_bar.png')

In [ ]:
from IPython.display import Image
Image(filename='../figures/cost_vs_f1.png')

## Phase 3 — Verdict (responde las 3 preguntas)

1. **¿Mejor equilibrio precisión-generalización (F1)? KNN (test F1-macro 0.995, CV 0.995±0.001).** CV≈test → generaliza, no es partición afortunada. LDA segundo (0.948) y 15× más rápido en fit. AdaBoost tercero (0.898) pese a ser el más complejo. NB último (0.579): la independencia no se cumple (128 sensores correlacionados) y confunde Acetaldehyde/Toluene (ver `cm_GaussianNB.png`).
2. **¿El complejo justifica su coste? No.** AdaBoost tarda 125 s (GridSearch) vs 0.4 s LDA / 6.3 s KNN / 0.3 s NB y rinde peor. Mejor costo/rendimiento: **LDA**. NB es el más barato pero inútil aquí. KNN tiene el predict más lento (1.8 s en test) por ser lazy O(n·d).
3. **¿Fronteras vs dimensionalidad (128-D, `boundary_*.png` en PCA-2D solo visualización)?** KNN → islas locales flexibles (captura clusters de gases, por eso gana con n=13k denso). LDA → rectas limpias (clases aprox. gaussianas separables). NB → elipses ingenuas solapadas (falla). AdaBoost → franjas escalonadas de stumps (sobre-fragmenta, sobreajusta al drift). En alta-D, el lineal simple (LDA) generaliza mejor que el ingenuo (NB); KNN gana porque n grande compensa la dimensionalidad.

**Nota metodológica:** los `boundary_*.png` se generan reentrenando un clon del mejor estimador sobre 2 componentes PCA (ver `src/experiment.py`) — son ilustración 2D, no la frontera real 128-D usada en las métricas.